# Lagged flux–OHC correlation over the full period 0001–0350 (backup)

Supplement to `17_ohc_corr_compare_analysis`, which runs the 0001–0220 and 0221–0350 windows: the same lag
correlation over the whole 0001–0350 period, kept in its own notebook so notebook 17 stays as is.

Figures:
* `lagcorr_qnet_ohc_0001-0350.pdf` — Q̄net, lags 0–200 (ticks every 5), panels (g)–(i)
* `lagcorr_qsource_ohc_0001-0350.pdf` — Q̄source (Qsw + Qlw_dn, notebook 5 definition), lags 0–100 (ticks every 10), panels (d)–(f)

Same classes and correlation settings as notebook 17's lag-correlation steps, with `ts_window = (1, 350)`,
`figsize = (12, 10)`, `fontsize = 40` and Full-CPL as the first panel label.

**Inputs:** `INPUT_DIR` = `data/ohc_corr/` — the OHC Hovmöller and annual flux files written by notebook 17
(steps 1 and 2.0) — **run notebook 17 first**. **Outputs:** figures to `FIG_DIR` (`figures/ohc_corr_full_period/`),
correlation NetCDFs to `DATA_DIR` (`data/ohc_corr_full_period/`).


In [ ]:
import os, sys
WF_ROOT = os.path.abspath("..")            # coupled_spinup_eval/
if WF_ROOT not in sys.path:
    sys.path.insert(0, WF_ROOT)
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
%matplotlib inline

import os, glob, re
import cftime
import numpy as np
import xarray as xr
from scipy import signal
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FixedFormatter, MultipleLocator, NullFormatter
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mticker
from typing import Union, Optional
from datetime import datetime, timezone
from typing import Dict, Sequence, Optional
from datetime import datetime

from util.experiments import build_exp_subdirs_from_run_dict, make_run_dict
from util.figures import publish
from util.regional_timeseries import LaggedOHCCorrelation, OHCHovmollerPlotter, write_annual_flux

## Setup

In [ ]:
# ---- paths ----
MODEL_ROOT   = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings"     # case directories
OUTPUT_ROOT  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3spinup_paper"   # published figures/ and data/
OUTPUT_URL   = "https://web.lcrc.anl.gov/public/e3sm/diagnostic_output/ac.szhang/v3spinup_paper"
FIG_DIR      = f"{OUTPUT_ROOT}/figures/ohc_corr_full_period"
DATA_DIR     = f"{OUTPUT_ROOT}/data/ohc_corr_full_period"  # cached/derived data (index: data/README.md)
INPUT_DIR    = f"{OUTPUT_ROOT}/data/ohc_corr"              # ohc_hov_* and FLUX_annual_* written by notebook 17

# ---- experiments: case directory names (under MODEL_ROOT) and model-year periods ----
EXPERIMENTS = {
    "Full-CPL": {
        "spin-up":   {"name": "20231209.v3.LR.piControl-spinup.chrysalis",      "period": "0001-2000"},
        "piControl": {"name": "v3.LR.piControl",                                "period": "0001-0500"},
    },
    "AltBG1": {
        "spin-up":   {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
    "AltBG2": {
        "spin-up":   {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
    "AltBG3": {
        "spin-up":   {"name": "20241105.v3.LR.piControl.Rerun.BC2GC.anvil",     "period": "0001-2000"},
        "piControl": {"name": "20241105.v3.LR.piControl.Rerun.BC2GC.anvil",     "period": "0001-0500"},
    },
}
LABEL = {"Full-CPL": "FC", "AltBG1": "FC-FOSI-S1", "AltBG2": "FC-FOSI-S2", "AltBG3": "FC-FOSI-S3"}

# ---- analysis settings ----
EXPS        = ["Full-CPL", "AltBG1", "AltBG2"]   # experiments compared (plotting order)
YEARS       = (1, 350)      # simulation years analysed (selects the MPAS-Analysis ts_0001-0350 run)
CLIMO_LEN   = 50            # climatology length of that MPAS-Analysis run (climo_0301-0350)


In [ ]:
# Shared inputs/outputs
run_dict = make_run_dict(EXPERIMENTS, MODEL_ROOT)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
publish(OUTPUT_ROOT, FIG_DIR)
print("figures ->", FIG_DIR)
print("data    ->", DATA_DIR)
print("inputs  ->", INPUT_DIR)
print("web     ->", FIG_DIR.replace(OUTPUT_ROOT, OUTPUT_URL))

## 1 Q̄net, 0001–0350 (notebook 17 step 2.2 with the full window)
### Settings

In [ ]:
# === Paths ===
data_dir    = INPUT_DIR  # ohc_hov_* and FLUX_annual_* from notebook 17
mpas_depth  = f"{OUTPUT_ROOT}/data/grid/depth.nc"
ts_base = "post/analysis/mpas_analysis"

fig_dir = FIG_DIR  # or os.path.join(results_dir, "figures")
os.makedirs(fig_dir, exist_ok=True)

# Optional window for pretty axis ticks if you later add custom plotting
ts_window   = (1, 350)
year_min, year_max = ts_window
year_token = f"{year_min:04d}-{year_max:04d}"
year_min, year_max = year_min-1, year_max+1

# === Your files ===
ohc_files = {
    "Full-CPL":
        f"{data_dir}/ohc_hov_20231209.v3.LR.piControl-spinup.chrysalis_total_annual_0001-0350.nc",
    "FC-FOSI-S1":
        f"{data_dir}/ohc_hov_20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis_total_annual_0001-0350.nc",
    "FC-FOSI-S2":
        f"{data_dir}/ohc_hov_20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis_total_annual_0001-0350.nc",
}
qnet_files = {
    "Full-CPL":
        f"{data_dir}/FLUX_annual_20231209.v3.LR.piControl-spinup.chrysalis_Qnet_Global_yr1-350.nc",
    "FC-FOSI-S1":
        f"{data_dir}/FLUX_annual_20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis_Qnet_Global_yr1-350.nc",
    "FC-FOSI-S2":
        f"{data_dir}/FLUX_annual_20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis_Qnet_Global_yr1-350.nc",
}

# --- Experiments (in desired plotting order) ---
exp_tags = list(ohc_files.keys())
labels   = list(ohc_files.keys())

lags = list(np.linspace(0, 200, 201, dtype=int))   # full period: lags to 200 (notebook 17: 100) #[0, 2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
vmax = 0.5
make_plots = True
ylabel = "Depth"
xlabel = "Lag (years)"
row_title = r"$r\left(\bar{Q}_{\mathrm{net}},\ \Delta\mathrm{OHC}\right)$"
figsize = (12, 10)     # full-period figures (notebook 17: (18, 16))
fontsize = 40          # (notebook 17: 60)

# If your files use different names, keep these overrides; otherwise set to None to auto-detect
ohc_var    = "ohc"
qnet_var   = "qnet"
time_name  = None
depth_name = "depth"
detrend = False
decode_times = False
fig_idx = 6            # panels (g)-(i)
xtick_int = 5


### Compute & plot

In [ ]:
corr_engine = LaggedOHCCorrelation(
    ohc_files, qnet_files,
    ohc_var=ohc_var,
    qnet_var=qnet_var,
    time_name=time_name,
    depth_name=depth_name,
    detrend=detrend,
    decode_times=decode_times
)

save_nc = os.path.join(DATA_DIR, f"lagcorr_{qnet_var}_ohc_bydepth_{year_token}.nc")  # <-- removed trailing comma
fig_name = f"lagcorr_qnet_ohc_{year_token}.pdf"
ds_corr = corr_engine.run(
    lags=lags,
    save_nc=save_nc,
    ts_window=ts_window,   # <--- just pass the tuple you already defined
    make_plots=make_plots,
    fig_dir=fig_dir,
    fig_name=fig_name,
    fig_idx=fig_idx,
    vmax=vmax,
    row_title=row_title,
    xtick_interval=xtick_int,  
    ylabel=ylabel,
    xlabel=xlabel,
    fontsize=fontsize,
    figsize=figsize
)

print(ds_corr)

publish(FIG_DIR, DATA_DIR)
print("web:", FIG_DIR.replace(OUTPUT_ROOT, OUTPUT_URL))


## 2 Q̄source, 0001–0350 (notebook 17 step 2.1 with the full window)
### Settings

In [ ]:
# === Paths ===
data_dir    = INPUT_DIR  # ohc_hov_* and FLUX_annual_* from notebook 17
mpas_depth  = f"{OUTPUT_ROOT}/data/grid/depth.nc"
ts_base = "post/analysis/mpas_analysis"

fig_dir = FIG_DIR  # or os.path.join(results_dir, "figures")
os.makedirs(fig_dir, exist_ok=True)

# Optional window for pretty axis ticks if you later add custom plotting
ts_window   = (1, 350)
year_min, year_max = ts_window
year_token = f"{year_min:04d}-{year_max:04d}"
year_min, year_max = year_min-1, year_max+1

# === Your files ===
ohc_files = {
    "Full-CPL":
        f"{data_dir}/ohc_hov_20231209.v3.LR.piControl-spinup.chrysalis_total_annual_0001-0350.nc",
    "FC-FOSI-S1":
        f"{data_dir}/ohc_hov_20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis_total_annual_0001-0350.nc",
    "FC-FOSI-S2":
        f"{data_dir}/ohc_hov_20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis_total_annual_0001-0350.nc",
}
qnet_files = {
    "Full-CPL":
        f"{data_dir}/FLUX_annual_20231209.v3.LR.piControl-spinup.chrysalis_Qsource_Global_yr1-350.nc",
    "FC-FOSI-S1":
        f"{data_dir}/FLUX_annual_20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis_Qsource_Global_yr1-350.nc",
    "FC-FOSI-S2":
        f"{data_dir}/FLUX_annual_20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis_Qsource_Global_yr1-350.nc",
}

# --- Experiments (in desired plotting order) ---
exp_tags = list(ohc_files.keys())
labels   = list(ohc_files.keys())

lags = list(np.linspace(0, 100, 101, dtype=int)) #[0, 2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
vmax = 0.5
make_plots = True
ylabel = "Depth"
xlabel = "Lag (years)"
row_title = r"$r\left(\bar{Q}_{\mathrm{source}},\ \Delta\mathrm{OHC}\right)$"
figsize = (12, 10)     # full-period figures (notebook 17: (18, 16))
fontsize = 40          # (notebook 17: 60)

# If your files use different names, keep these overrides; otherwise set to None to auto-detect
ohc_var    = "ohc"
qnet_var   = "qsource"
time_name  = None
depth_name = "depth"
detrend = False
decode_times = False
fig_idx = 3            # panels (d)-(f)
xtick_int = 10


### Compute & plot

In [ ]:
corr_engine = LaggedOHCCorrelation(
    ohc_files, qnet_files,
    ohc_var=ohc_var,
    qnet_var=qnet_var,
    time_name=time_name,
    depth_name=depth_name,
    detrend=detrend,
    decode_times=decode_times
)

save_nc = os.path.join(DATA_DIR, f"lagcorr_{qnet_var}_ohc_bydepth_{year_token}.nc")  # <-- removed trailing comma
fig_name = f"lagcorr_qsource_ohc_{year_token}.pdf"
ds_corr = corr_engine.run(
    lags=lags,
    save_nc=save_nc,
    ts_window=ts_window,   # <--- just pass the tuple you already defined
    make_plots=make_plots,
    fig_dir=fig_dir,
    fig_name=fig_name,
    fig_idx=fig_idx,
    vmax=vmax,
    row_title=row_title,
    xtick_interval=xtick_int,  
    ylabel=ylabel,
    xlabel=xlabel,
    fontsize=fontsize,
    figsize=figsize
)

print(ds_corr)

publish(FIG_DIR, DATA_DIR)
print("web:", FIG_DIR.replace(OUTPUT_ROOT, OUTPUT_URL))
